In [ ]:
import pandas as pd
import numpy as np
import os
import pydicom
import cv2
import shutil
from concurrent.futures import ThreadPoolExecutor

def apply_windowing(img, window_center, window_width):
    """Applies windowing to the DICOM image using the Window Center and Window Width."""
    img_min = window_center - (window_width / 2)
    img_max = window_center + (window_width / 2)
    # Clip the image and normalize to range 0-255
    img = np.clip(img, img_min, img_max)
    img = ((img - img_min) / (img_max - img_min) * 255).astype('uint8')
    return img

def convert(f, output_path):
    try:
        # Read the DICOM image
        ds = pydicom.dcmread(f)
        img = ds.pixel_array

        # Get Window Center and Window Width from DICOM metadata, if available
        window_center = ds.WindowCenter if 'WindowCenter' in ds else None
        window_width = ds.WindowWidth if 'WindowWidth' in ds else None
        
        # If windowing data is available, apply windowing
        if window_center and window_width:
            if isinstance(window_center, pydicom.multival.MultiValue):
                window_center = window_center[0]
            if isinstance(window_width, pydicom.multival.MultiValue):
                window_width = window_width[0]
            img = apply_windowing(img, float(window_center), float(window_width))
        else:
            # If no windowing data, normalize the image to the range 0-255
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

        # Resize the image to 250x250 pixels
        img = cv2.resize(img, (250, 250))

        # Save the image as a .jpg file
        cv2.imwrite(output_path, img)

    except Exception as e:
        print(f"Error converting {f}: {e}")

# Load the CSV files
label = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_label_coordinates.csv")
train = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv")

# Check if directories exist, if not, create them
if not os.path.exists("./kaggle/working/train_wolf"):
    os.makedirs("./kaggle/working/train_wolf/spinal_canal_stenosis/normal")
    os.makedirs("./kaggle/working/train_wolf/spinal_canal_stenosis/Moderate")
    os.makedirs("./kaggle/working/train_wolf/spinal_canal_stenosis/Severe")
    os.makedirs("./kaggle/working/train_wolf/left_neural_foraminal_narrowing/normal")
    os.makedirs("./kaggle/working/train_wolf/left_neural_foraminal_narrowing/Moderate")
    os.makedirs("./kaggle/working/train_wolf/left_neural_foraminal_narrowing/Severe")
    os.makedirs("./kaggle/working/train_wolf/right_neural_foraminal_narrowing/normal")
    os.makedirs("./kaggle/working/train_wolf/right_neural_foraminal_narrowing/Moderate")
    os.makedirs("./kaggle/working/train_wolf/right_neural_foraminal_narrowing/Severe")
    os.makedirs("./kaggle/working/train_wolf/left_subarticular_stenosis/normal")
    os.makedirs("./kaggle/working/train_wolf/left_subarticular_stenosis/Moderate")
    os.makedirs("./kaggle/working/train_wolf/left_subarticular_stenosis/Severe")
    os.makedirs("./kaggle/working/train_wolf/right_subarticular_stenosis/normal")
    os.makedirs("./kaggle/working/train_wolf/right_subarticular_stenosis/Moderate")
    os.makedirs("./kaggle/working/train_wolf/right_subarticular_stenosis/Severe")

# Function to process each image
def process_image(i):
    condition = i[3]
    level = i[4]
    study_id = i[0]
    series_id = i[1]
    instance_number = i[2]

    w_ = f"{'_'.join(condition.split(' '))}_{'_'.join(level.split('/'))}".lower()
    wolf = train[train["study_id"] == study_id].to_dict('list')[w_][0]
    
    if wolf == "Normal/Mild":
        wolf = "normal"

    output_path = f"./kaggle/working/train_wolf/{condition.lower().replace(' ', '_')}/{wolf}/{i[5]}_{w_}.jpg"
    dicom_path = f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/{study_id}/{series_id}/{instance_number}.dcm"

    convert(dicom_path, output_path)

# Process images in parallel using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_image, zip(np.array(label["study_id"]),
                                    np.array(label["series_id"]),
                                    np.array(label['instance_number']),
                                    np.array(label['condition']),
                                    np.array(label['level']),
                                    range(len(np.array(label['level'])))))


In [ ]:
!rm -rf /kaggle/working/*

In [3]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation




def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Discriminator(nn.Module):
	def __init__(self):
		super().__init__()
		self.devil=nn.Sequential(
			nn.Conv2d(3, 4,3),
			nn.BatchNorm2d(4),
			nn.ReLU(),
			nn.AvgPool2d(2, 2),

			nn.Conv2d(4, 8, 3),
			nn.BatchNorm2d(8),
			nn.AvgPool2d(2, 2),

			nn.Conv2d(8, 16, 3),
			nn.BatchNorm2d(16),
			nn.AvgPool2d(2, 2),

			nn.Conv2d(16, 32, 3),
			nn.BatchNorm2d(32),
			nn.AvgPool2d(2, 2),

			nn.Conv2d(32, 64, 3),
			nn.BatchNorm2d(64),
			nn.AvgPool2d(2, 2),

			nn.Conv2d(64, 128, 3),
			nn.BatchNorm2d(128),
			nn.AvgPool2d(2, 2),

			nn.Flatten(),

			nn.Linear(512, 64),
			nn.Linear(64, 32),
			nn.Linear(32, 16),
			nn.Linear(16, 3)
			)

	def forward(self,x):
		return self.devil(x)




if __name__=="__main__":
	torch.backends.cudnn.benchmark = True
	z_w=[]
	manualSeed = 999
	print("Random Seed: ", manualSeed)
	random.seed(manualSeed)
	torch.manual_seed(manualSeed)
	torch.use_deterministic_algorithms(True)
	os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

		# The flag below controls whether to allow TF32 on matmul. This flag defaults to False
	# in PyTorch 1.12 and later.
	torch.backends.cuda.matmul.allow_tf32 = True

	# The flag below controls whether to allow TF32 on cuDNN. This flag defaults to True.
	torch.backends.cudnn.allow_tf32 = True
	workers = 12
	batch_size = 128
	nz = 100
	num_epochs = 5
	lr = 0.001
	beta1 = 0.5
	ngpu=1
	ngf,nc = 3,3
	ndf = 64
	nttepoch=0
	#transforms.Resize(size=(config.INPUT_HEIGHT,config.INPUT_WIDTH))
	transform = transforms.Compose(
		[transforms.Resize(256),transforms.ToTensor(),
		transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

	trainset = torchvision.datasets.ImageFolder(root=f"./kaggle/working/train_wolf",transform=transform)
	dataloader = torch.utils.data.DataLoader(trainset,batch_size=batch_size,shuffle=True,num_workers=workers)
	#testset_ = torchvision.datasets.ImageFolder(root=f"E:/rsna/neural_foraminal_narrowing",transform=transform)
	#dataloader_ = torch.utils.data.DataLoader(testset_,batch_size=,shuffle=True,num_workers=32)
	device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

	netD = Discriminator().to(device)
	if (device.type == 'cuda') and (ngpu > 1):
		netD = nn.DataParallel(netD, list(range(ngpu)))
	netD.apply(weights_init)
	try:
		netD.load_state_dict(torch.load("./best-model.pt"))
		print("Loaded")
	except:
		pass
	#netD.load_state_dict(torch.load("E:/rsna/best-model.pt", map_location=device))

	criterion,img_devil = nn.CrossEntropyLoss(),0
	fixed_noise = torch.randn(1, nz, 1, 1, device=device)

	optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
	schedulerD=torch.optim.lr_scheduler.ExponentialLR(optimizerD, gamma=0.86)
	#schedulerD=torch.optim.lr_scheduler.OneCycleLR(optimizerD)
	print(dataloader.dataset.classes)


	img_list = []
	G_losses = []
	D_losses = []
	iters,z_ = 0,0
	lr_i=0
	print("Starting Training Loop...")
	best_performance = 0  # Variable to track the best performance so far
	model_path = "E:/rsna/best-model.pt" # Path to save the best model
	least_loss=float('inf')
	while(True):
		epoch_loss = 0.0
		z_, z,class_0c,class_1c,class2_c,class_0w,class_1w,class2_w = 0, 0,0,0,0,0,0,0

		for i, data in enumerate(dataloader, 0):
			nttepoch+=1
			optimizerD.zero_grad() #Added line
			real_cpu = data[0].to(device)
			label = data[1].to(device)
			output = netD(real_cpu)
			errD_real = criterion(output, label)
			epoch_loss+=errD_real.item()
			errD_real.backward()
			optimizerD.step()
			label_=label
			output = torch.argmax(netD(real_cpu),dim=1).view(-1)
			#print(output)
			correct_arr=(label.detach().cpu().numpy().reshape(-1) == output.detach().cpu().numpy())
			z+=correct_arr.sum()
			for i,j in zip(output.detach().cpu().numpy(),correct_arr):
				if j:
					if i==0:
						class_0c+=1
					elif i==1:
						class_1c+=1
					elif i==2:
						class2_c+=1
				else:
					if i==0:
						class_0w+=1
					elif i==1:
						class_1w+=1
					elif i==2:
						class2_w+=1
			z_ += len(label)
			if(nttepoch==1):
				print("Label:",label)
				print("Output:",output)
				print("label.detach().cpu().numpy().reshape(-1) == output.detach().cpu().numpy():")
				print(label.detach().cpu().numpy().reshape(-1))
				print(output.detach().cpu().numpy())
		print(f"Loss:{epoch_loss/len(dataloader)} \nClass 0: {class_0c}/{class_0w+class_0c} Class 1: {class_1c}/{class_1c+class_1w} Class 2: {class2_c}/{class2_w+class2_c}")
		
		print("Sample went throught:",z_,"Classified correctly:",z)
		z_w.append(z)
		if len(z_w)>=3:
			if len([True for i in range(1,3) if z_w[len(z_w)-i]<=z_w[len(z_w)-3]+2 and z_w[len(z_w)-i]>=z_w[len(z_w)-4]-3])==2:
				print([True for i in range(1,3) if z_w[len(z_w)-i]<=z_w[len(z_w)-3]+2 and z_w[len(z_w)-i]>=z_w[len(z_w)-4]-3])
				z_w=[]
				print(optimizerD.param_groups[0]["lr"])
				schedulerD.step()
				print(optimizerD.param_groups[0]["lr"])
		if z > best_performance or (z == best_performance and epoch_loss/len(dataloader) < least_loss):
			best_performance = z
			least_loss = epoch_loss/len(dataloader)
			torch.save(netD.state_dict(), model_path)
			print(f"Best model saved with performance: {best_performance} and loss: {least_loss} \nClass 1: {class_1c}/{class_1c+class_1w} Class 2: {class2_c}/{class2_w+class2_c}")

Random Seed:  999


/opt/conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


['left_neural_foraminal_narrowing', 'left_subarticular_stenosis', 'right_neural_foraminal_narrowing', 'right_subarticular_stenosis', 'spinal_canal_stenosis']
Starting Training Loop...


/tmp/ipykernel_31/2084510718.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  netD.load_state_dict(torch.load("./best-model.pt"))


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 309, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/opt/conda/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
  File "/opt/conda/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 317, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/opt/conda/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 174, in collate
    return [collate(samples, collate_fn_map=collate_fn_map) for samples in transposed]  # Backwards compatibility.
  File "/opt/conda/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 174, in <listcomp>
    return [collate(samples, collate_fn_map=collate_fn_map) for sampl

In [ ]:
import os
print(os.cpu_count())  # This returns the number of CPU cores
